In [1]:
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
import numpy

from tqdm.contrib.concurrent import process_map

from lightcurves import features, LC_DATA_PATH

In [2]:
classes = ('lensing', 'not-lensing')

xs = []
ys = []

for y, c in enumerate(classes):
    paths = list((LC_DATA_PATH / c).glob('*.fits.gz'))
    xs_c = process_map(features, paths, desc=f"Loading {c}", chunksize=100)
    xs.append(xs_c)
    ys += [y] * len(xs_c)

xs = numpy.concatenate(xs)
ys = numpy.array(ys)
        

Loading lensing:   0%|          | 0/8986 [00:00<?, ?it/s]

Loading not-lensing:   0%|          | 0/15310 [00:00<?, ?it/s]

In [3]:
x_train, x_test, y_train, y_test = train_test_split(xs, ys, test_size=.2)

In [4]:
bst = XGBClassifier(n_estimators=2, max_depth=2, learning_rate=1, objective='binary:logistic')
# fit model
bst.fit(x_train, y_train)
# make predictions
preds = bst.predict(x_test)

In [5]:
(preds == y_test).mean()

np.float64(0.8895061728395062)